This notebook will help set up parameters for SBS by asessing the following:
- Is error correction appropiate?

This notebook will also generate a scrambled barcode library to test whether SBS parameters are capturing noise or real signal.

# Imports

In [2]:
import pandas as pd
import numpy as np
from itertools import combinations
import Levenshtein

# Error Correction

In [3]:

def barcode_distance_matrix(barcodes_1, barcodes_2=False, distance_metric="hamming"):
    """Calculate distances between two sets of barcodes.

    Creates a matrix of distances between all pairs of barcodes from two sets.
    If only one set is provided, computes self-distances.

    Args:
        barcodes_1 (list): First list of barcode sequences
        barcodes_2 (list or bool, optional): Second list of barcode sequences.
            If False, uses barcodes_1 for both sets. Default is False.
        distance_metric (str, optional): Type of distance to calculate.
            Options are 'hamming' or 'levenshtein'. Default is 'hamming'.

    Returns:
        numpy.ndarray: Matrix of distances between barcode pairs
    """
    import warnings

    # Define the distance function based on chosen metric
    if distance_metric == "hamming":
        distance = lambda i, j: Levenshtein.hamming(i, j)
    elif distance_metric == "levenshtein":
        distance = lambda i, j: Levenshtein.distance(i, j)
    else:
        warnings.warn(
            'distance_metric must be "hamming" or "levenshtein" - defaulting to "hamming"'
        )
        distance = lambda i, j: Levenshtein.hamming(i, j)

    # If second set not provided, use the first set
    if isinstance(barcodes_2, bool):
        barcodes_2 = barcodes_1

    # Create distance matrix for all barcode pairs
    bc_distance_matrix = np.zeros((len(barcodes_1), len(barcodes_2)))
    for a, i in enumerate(barcodes_1):
        for b, j in enumerate(barcodes_2):
            bc_distance_matrix[a, b] = distance(i, j)

    return bc_distance_matrix

def check_error_correction_suitability(
    barcode_library, 
    barcode_col="prefix_map", 
    max_distance=2,
    distance_metric="hamming",
    sample_size=1000
):
    """
    Check if a barcode library is suitable for error correction.
    
    Args:
        barcode_library (pd.DataFrame or str): DataFrame containing barcode sequences, 
                                              or path to TSV file
        barcode_col (str): Column name containing the barcode sequences
        max_distance (int): Maximum distance for error correction (default: 2)
        distance_metric (str): Distance metric to use ('hamming' or 'levenshtein')
        sample_size (int): Number of barcode pairs to sample for distance analysis
    
    Returns:
        dict: Analysis results and recommendations
    """
    # Handle both DataFrame and file path inputs
    if isinstance(barcode_library, str):
        # It's a file path, load it
        barcode_library = pd.read_csv(barcode_library, sep='\t')
    elif not isinstance(barcode_library, pd.DataFrame):
        return {"error": "barcode_library must be a pandas DataFrame or file path string"}
    
    barcodes = barcode_library[barcode_col].dropna().unique()
    n_barcodes = len(barcodes)
    
    if n_barcodes == 0:
        return {"error": "No barcodes found in the specified column"}
    
    # Basic statistics
    barcode_lengths = [len(bc) for bc in barcodes]
    min_length = min(barcode_lengths)
    max_length = max(barcode_lengths)
    avg_length = np.mean(barcode_lengths)
    
    results = {
        "n_barcodes": n_barcodes,
        "barcode_length_range": (min_length, max_length),
        "avg_barcode_length": avg_length,
        "max_distance": max_distance,
        "distance_metric": distance_metric
    }
    
    # Check for variable length barcodes
    if min_length != max_length:
        results["warning"] = f"Variable barcode lengths detected ({min_length}-{max_length}). Error correction may be unreliable."
    
    # Theoretical maximum possible unique sequences for given length
    if distance_metric == "hamming" and min_length == max_length:
        # For fixed-length sequences, calculate theoretical capacity
        theoretical_max = 4 ** min_length  # assuming DNA sequences
        results["theoretical_max_sequences"] = theoretical_max
        results["library_density"] = n_barcodes / theoretical_max
    
    # Sample pairwise distances to assess library design
    if n_barcodes > 1:
        # For large libraries, sample pairs to avoid computational explosion
        if n_barcodes > 100:
            # Sample pairs randomly
            pair_indices = np.random.choice(n_barcodes, size=min(sample_size, n_barcodes), replace=False)
            sample_barcodes = [barcodes[i] for i in pair_indices]
        else:
            sample_barcodes = barcodes
        
        # Calculate distances between sampled barcodes
        distances = barcode_distance_matrix(
            sample_barcodes, 
            sample_barcodes, 
            distance_metric=distance_metric
        )
        
        # Remove diagonal (self-distances)
        mask = np.ones(distances.shape, dtype=bool)
        np.fill_diagonal(mask, False)
        pairwise_distances = distances[mask]
        
        results["min_pairwise_distance"] = np.min(pairwise_distances)
        results["mean_pairwise_distance"] = np.mean(pairwise_distances)
        results["distances_within_max_distance"] = np.sum(pairwise_distances <= max_distance)
        results["fraction_close_pairs"] = np.sum(pairwise_distances <= max_distance) / len(pairwise_distances)
    
    # Generate recommendations
    recommendations = []
    
    # Check if barcodes are too short
    if avg_length <= max_distance + 1:
        recommendations.append(
            f"⚠️  CRITICAL: Average barcode length ({avg_length:.1f}) is too short for "
            f"max_distance={max_distance}. Consider longer barcodes or smaller max_distance."
        )
    
    # Check minimum distance between barcodes
    if "min_pairwise_distance" in results:
        min_dist = results["min_pairwise_distance"]
        recommended_min_dist = 2 * max_distance + 1
        
        if min_dist < recommended_min_dist:
            recommendations.append(
                f"⚠️  WARNING: Minimum distance between barcodes ({min_dist}) is less than "
                f"recommended minimum ({recommended_min_dist}) for reliable error correction."
            )
        
        if min_dist <= max_distance:
            recommendations.append(
                f"🚫 CRITICAL: Some barcodes are within max_distance ({max_distance}) of each other. "
                f"Error correction will be unreliable or impossible for these sequences."
            )
    
    # Check fraction of close pairs
    if "fraction_close_pairs" in results:
        if results["fraction_close_pairs"] > 0.1:
            recommendations.append(
                f"⚠️  WARNING: {results['fraction_close_pairs']:.1%} of barcode pairs are within "
                f"max_distance={max_distance}. High potential for correction ambiguity."
            )
    
    # Library density check
    if "library_density" in results:
        if results["library_density"] > 0.1:
            recommendations.append(
                f"⚠️  WARNING: Library uses {results['library_density']:.1%} of theoretical "
                f"sequence space. Dense packing may cause correction conflicts."
            )
    
    # Generate overall recommendation
    if not recommendations:
        overall_recommendation = "✅ Library appears suitable for error correction with current parameters."
    elif any("CRITICAL" in rec for rec in recommendations):
        overall_recommendation = "🚫 Library NOT suitable for error correction with current parameters."
    else:
        overall_recommendation = "⚠️  Library may work for error correction but consider adjusting parameters."
    
    results["recommendations"] = recommendations
    results["overall_recommendation"] = overall_recommendation
    
    return results

def print_error_correction_analysis(analysis_results):
    """Print a formatted report of the error correction analysis."""
    
    print("=" * 60)
    print("BARCODE ERROR CORRECTION SUITABILITY ANALYSIS")
    print("=" * 60)
    
    # Basic stats
    print(f"Number of unique barcodes: {analysis_results['n_barcodes']}")
    print(f"Barcode length range: {analysis_results['barcode_length_range'][0]}-{analysis_results['barcode_length_range'][1]} bp")
    print(f"Average barcode length: {analysis_results['avg_barcode_length']:.1f} bp")
    print(f"Max correction distance: {analysis_results['max_distance']}")
    print(f"Distance metric: {analysis_results['distance_metric']}")
    
    if "min_pairwise_distance" in analysis_results:
        print(f"Minimum distance between barcodes: {analysis_results['min_pairwise_distance']}")
        print(f"Mean pairwise distance: {analysis_results['mean_pairwise_distance']:.1f}")
        print(f"Fraction of pairs within max_distance: {analysis_results['fraction_close_pairs']:.1%}")
    
    print("\n" + "=" * 60)
    print("RECOMMENDATIONS")
    print("=" * 60)
    
    print(analysis_results['overall_recommendation'])
    print()
    
    for rec in analysis_results['recommendations']:
        print(rec)
    
    if not analysis_results['recommendations']:
        print("No specific warnings or recommendations.")


In [11]:
# Example usage:
# First load your barcode library
df_barcode_library = pd.read_csv("/lab/ops_analysis/lourido/nebo-analysis/analysis/config/barcode_library.tsv", sep='\t')
analysis = check_error_correction_suitability(df_barcode_library, "prefix_map", max_distance=2)
print_error_correction_analysis(analysis)

BARCODE ERROR CORRECTION SUITABILITY ANALYSIS
Number of unique barcodes: 93
Barcode length range: 7-7 bp
Average barcode length: 7.0 bp
Max correction distance: 2
Distance metric: hamming
Minimum distance between barcodes: 3.0
Mean pairwise distance: 5.0
Fraction of pairs within max_distance: 0.0%

RECOMMENDATIONS
⚠️  Library may work for error correction but consider adjusting parameters.

⚠️  WARNING: Minimum distance between barcodes (3.0) is less than recommended minimum (5) for reliable error correction.


In [5]:
# Check error correction requirements at different cycle lengths
library_design_df = pd.read_csv("/home/acepedadiaz/CROPseq_multi_main/CROPseq-multi/oligo_designs/CSM_library_design.20250825-110847.csv")
for cycles in range(8, 13):  # Check 8, 9, 10, 11, 12 cycles
    # Truncate barcodes to this length
    truncated_barcodes = library_design_df['iBAR_2'].str[:cycles]
    
    # Calculate minimum pairwise distance
    distances = barcode_distance_matrix(truncated_barcodes.values, distance_metric='hamming')
    
    # Remove diagonal (self-distances)
    mask = np.ones(distances.shape, dtype=bool)
    np.fill_diagonal(mask, False)
    min_distance = np.min(distances[mask])
    
    print(f"{cycles} cycles: minimum distance = {min_distance}")
    if min_distance >= 3:
        print(f"✅ Error correction possible at {cycles} cycles (+{cycles-8} additional cycles)")
        break
    else:
        print(f"❌ Error correction NOT reliable at {cycles} cycles")

8 cycles: minimum distance = 1.0
❌ Error correction NOT reliable at 8 cycles
9 cycles: minimum distance = 2.0
❌ Error correction NOT reliable at 9 cycles
10 cycles: minimum distance = 3.0
✅ Error correction possible at 10 cycles (+2 additional cycles)


# Library scramble

In [6]:
def create_derangement(arr, rng, max_attempts=1000):
    """
    Create a derangement (permutation where no element appears in its original position).
    Falls back to regular permutation with manual fixes if needed.
    """
    original = arr.copy()
    unique_values = len(set(arr))
    total_positions = len(arr)
    
    # Check if derangement is theoretically possible
    if unique_values == 1:
        raise ValueError(
            f"Cannot create scrambled barcodes: all {total_positions} barcodes are identical. "
            f"With only 1 unique barcode value, it's impossible to generate a scrambled set "
            f"that doesn't match the original positions."
        )
    
    if unique_values == 2 and total_positions == 2:
        # Special case: with 2 positions and 2 unique values, 
        # derangement is always possible by swapping
        pass
    elif unique_values < total_positions:
        # Check if the distribution makes derangement impossible
        value_counts = pd.Series(arr).value_counts()
        max_count = value_counts.max()
        
        # If any single value appears in more than half the positions,
        # and we have limited unique values, derangement becomes very difficult
        if max_count > total_positions // 2 and unique_values <= 3:
            print(f"Warning: {max_count} positions contain the same barcode out of {total_positions} total. "
                  f"With only {unique_values} unique barcodes, scrambling may be difficult.")
    
    for attempt in range(max_attempts):
        # Create a random permutation
        scrambled = rng.permutation(arr)
        
        # Check if it's a derangement (no element in original position)
        if not np.array_equal(scrambled, original) and not any(scrambled == original):
            return scrambled
    
    # If we can't find a perfect derangement, create one manually
    # This is a fallback for small arrays or edge cases
    scrambled = rng.permutation(arr)
    
    # Fix any elements that are in their original positions
    for i in range(len(scrambled)):
        if scrambled[i] == original[i]:
            # Find another position to swap with
            for j in range(len(scrambled)):
                if j != i and scrambled[j] != original[j] and scrambled[i] != original[j] and scrambled[j] != original[i]:
                    scrambled[i], scrambled[j] = scrambled[j], scrambled[i]
                    break
            else:
                # Could not find a valid swap - this means derangement is impossible
                unique_vals = len(set(original))
                raise ValueError(
                    f"Cannot create scrambled barcodes: given {total_positions} barcode positions "
                    f"with only {unique_vals} unique barcode values, it's impossible to generate "
                    f"a scrambled set where no barcode appears in its original position. "
                    f"Consider using a barcode library with more unique sequences."
                )
    
    return scrambled

def scramble_prefix_columns(input_tsv, output_tsv, seed=42):
    # Load TSV
    df = pd.read_csv(input_tsv, sep='\t')

    # Check required columns exist
    required_cols = ['prefix_map', 'gene_symbol', 'uniprot_entry', 'prefix_recomb']
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"Input TSV must contain columns: {required_cols}")

    # Set random seed for reproducibility
    rng = np.random.default_rng(seed)

    # Store original values for comparison
    original_prefix_map = df['prefix_map'].values.copy()
    original_prefix_recomb = df['prefix_recomb'].values.copy()

    # Create derangements (permutations with no fixed points)
    df['prefix_map'] = create_derangement(df['prefix_map'].values, rng)
    df['prefix_recomb'] = create_derangement(df['prefix_recomb'].values, rng)

    # Validation: ensure scrambling actually occurred
    map_changed = not np.array_equal(df['prefix_map'].values, original_prefix_map)
    recomb_changed = not np.array_equal(df['prefix_recomb'].values, original_prefix_recomb)
    
    if not map_changed or not recomb_changed:
        raise RuntimeError("Failed to create properly scrambled columns")
    
    # Additional validation: count how many positions changed
    map_changes = sum(df['prefix_map'].values != original_prefix_map)
    recomb_changes = sum(df['prefix_recomb'].values != original_prefix_recomb)
    
    print(f"prefix_map: {map_changes}/{len(df)} positions changed")
    print(f"prefix_recomb: {recomb_changes}/{len(df)} positions changed")

    # Save to file
    df.to_csv(output_tsv, sep='\t', index=False)
    print(f"Scrambled and validated columns saved to: {output_tsv}")

# Example usage:
scramble_prefix_columns("/lab/ops_analysis/lourido/nebo-analysis/analysis/config/old_barcode_library.tsv", 
                       "/lab/ops_analysis/lourido/nebo-analysis/analysis/config/barcode_library.tsv")

FileNotFoundError: [Errno 2] No such file or directory: '/lab/ops_analysis/lourido/nebo-analysis/analysis/config/old_barcode_library.tsv'